# Phase 5 — NB1: Stage 1 Category Detection (MAMS)

**Goal:** Train Stage 1 Category Detection on MAMS dataset with native 8 categories.

**Config:** `stage1_mams_cataware.yaml` — Cat-Aware Attention + BCE

**Dataset:** MAMS-ACSA (cloned from GitHub)
- Train: train.xml + val.xml merged (~3,549 sentences)
- Test: test.xml (400 sentences)
- 8 categories: ambience, food, menu, miscellaneous, place, price, service, staff

**Output:** Upload `/kaggle/working/outputs_p5_nb1_mams/` as Kaggle dataset `p5-nb1-stage1-mams`

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml iterative-stratification

In [ ]:
import os, sys, json, shutil

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
# Clone MAMS dataset from GitHub
!git clone https://github.com/siat-nlp/MAMS-for-ABSA.git data/mams

# Verify MAMS data files
for f in ['data/mams/data/MAMS-ACSA/raw/train.xml',
          'data/mams/data/MAMS-ACSA/raw/val.xml',
          'data/mams/data/MAMS-ACSA/raw/test.xml']:
    assert os.path.exists(f), f'MISSING: {f}'
    print(f'OK: {f}')
print('MAMS data ready.')

In [ ]:
import torch, gc
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Prepare MAMS Data (8 native categories)

In [ ]:
!python scripts/01_prepare_data_mams.py

In [ ]:
for f in ['category_detection.jsonl', 'sentiment_records.jsonl',
         'classification.jsonl', 'contrastive_triplets_polonly.jsonl']:
    path = f'data/processed_mams/{f}'
    if os.path.exists(path):
        with open(path) as fp:
            n = sum(1 for _ in fp)
        print(f'{f}: {n} records')
    else:
        print(f'MISSING: {path}')

# Verify 8 categories in category_detection
with open('data/processed_mams/category_detection.jsonl') as fp:
    first = json.loads(fp.readline())
print(f'\nCategory vector length: {len(first["category_vector"])} (expect 8)')
print(f'Sample categories: {first["categories"]}')

## 2. Train — Cat-Aware Attention (MAMS, 8 categories)

Config: `stage1_mams_cataware.yaml`
- 8 categories (native MAMS), Cat-Aware Attention + BCE
- encoder_lr=1e-5, head_lr=5e-4, 30 epochs, patience=5

In [ ]:
gc.collect()
torch.cuda.empty_cache()
!python scripts/04a_train_stage1.py --config configs/stage1_mams_cataware.yaml

In [ ]:
log_path = 'logs/stage1_mams_cataware_training.jsonl'
print('=== MAMS Cat-Aware Training Log ===')
if os.path.exists(log_path):
    print(f'{"Epoch":<8} {"Train Loss":<12} {"Val Loss":<12} {"Cat F1":<10} {"Cat P":<10} {"Cat R":<10}')
    print('-' * 64)
    with open(log_path) as f:
        for line in f:
            r = json.loads(line)
            print(f"{r['epoch']:<8} {r['train_loss']:<12.4f} {r.get('loss', 0):<12.4f} "
                  f"{r['category_f1']:<10.4f} "
                  f"{r.get('category_precision', 0):<10.4f} {r.get('category_recall', 0):<10.4f}")
else:
    print('No log found.')

## 3. Save Outputs

In [ ]:
output_dir = '/kaggle/working/outputs_p5_nb1_mams'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f'{output_dir}/logs', exist_ok=True)

# Checkpoint
src = 'checkpoints/stage1_mams_cataware/best.pt'
if os.path.exists(src):
    shutil.copy(src, f'{output_dir}/stage1_mams_cataware_best.pt')
    print(f'stage1_mams_cataware_best.pt: {os.path.getsize(src)/1e6:.1f} MB')
else:
    print(f'WARNING: {src} not found')

# Processed data
for fname in ['category_detection.jsonl', 'sentiment_records.jsonl',
              'classification.jsonl', 'contrastive_triplets_polonly.jsonl']:
    src = f'data/processed_mams/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{output_dir}/{fname}')
        print(f'{fname} copied')

# Log
log = 'logs/stage1_mams_cataware_training.jsonl'
if os.path.exists(log):
    shutil.copy(log, f'{output_dir}/logs/')
    print('Training log saved')

print(f'\nOutputs saved to {output_dir}')
print('Upload as Kaggle dataset: p5-nb1-stage1-mams')

In [ ]:
shutil.make_archive('/kaggle/working/outputs_p5_nb1_mams_backup', 'zip',
                    '/kaggle/working', 'outputs_p5_nb1_mams')
size_mb = os.path.getsize('/kaggle/working/outputs_p5_nb1_mams_backup.zip') / 1e6
print(f'Backup zip: {size_mb:.1f} MB')